# EUR_9A — V5 with label-free z-score statistics on CHB-MIT (Track F, chat D31-F)

**Design:** `results/eureka_d31f/STEP1_DESIGN.md` §3 (approved 19/11/2026 20:35). One change to V5: the per-patient
z-score statistics of the 23 CHB-MIT cases are computed by the record function **without** the seizure map (every 10th
window of every file), then the features are rebuilt by the record builder and scored with the four committed V5
checkpoints (k recalibrated by V5's rule, and at the record's k). The raw EDF are verified against PhysioNet's SHA256SUMS
in this session. **Runs a SMOKE first** (chb08: the record statistics and the first 2 files' features reproduce the
record) and the V5 control (R1.0) before the long build; STOPs on any difference.
**Accelerator:** None (CPU). **Internet:** ON (pip, git clone, PhysioNet checksums and missing files).
**Inputs (Add Input):** `combined-siena-scalp-and-chb-mit-eeg-datasets` (raw EDF), `v9-x3-fold1`, `v9-x3-fold3`.
**Output:** `/kaggle/working/eureka/9a` (→ `results/eureka_d31f/9a`; small files only). **Runtime:** ≈ 3–4 h (computed
from R0's measured statistics and feature rates; the run writes `projection.json` and STOPs if it projects > 10.5 h).
Code: `github.com/Nhanvas/Thesis` ref `main`; every module's sha256 is asserted below.
**Nothing printed here is a research result.** Do not edit; *Save Version → Save & Run All (Commit)*.

In [ ]:
# SETUP: packages + repository (read only)
import os, sys, subprocess, time, json, hashlib
T_NB = time.time()
NOTEBOOK, REPO_REF = "EUR_9A", "main"
TEST = bool(os.environ.get("EUR_TEST_REPO"))                     # synthetic test in the build container only
WORK = os.environ.get("EUR_TEST_WORK", "/kaggle/working")
INPUT = os.environ.get("EUR_TEST_INPUT", "/kaggle/input")
SCRATCH = os.environ.get("EUR_TEST_SCRATCH", "/tmp/r2")
if TEST:
    REPO = os.environ["EUR_TEST_REPO"]; HEAD = "test"
else:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "mne==1.12.1", "torch_geometric==2.7.0", "ruptures==1.1.10", "timescoring==0.0.7"])
    REPO = "/tmp/Thesis"
    if not os.path.exists(REPO + "/src/eur_9a.py"):
        subprocess.check_call(f"rm -rf {REPO} && git clone -q https://github.com/Nhanvas/Thesis.git {REPO} "
                              f"&& cd {REPO} && git checkout -q {REPO_REF}", shell=True)
    HEAD = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"]).decode().strip()
print("repository", REPO, HEAD, f"({time.time()-T_NB:.0f}s)")

In [ ]:
# START: sha256 of every module (the reviewed code only), environment, imports
WANT = {"eur_9a.py": "c1995d8b3dd393f4", "eur_abl.py": "d7da61331a4c098f", "r2_r1.py": "76cbe51ff0180d4a", "r2_core.py": "902d15ad4feb3e8a", "e16_recording_baseline.py": "f170e1c59f5b0447", "e17b_latent_family.py": "3b4f034dca00eaf3", "p3c_freeze.py": "dabe4139f6005fd2", "p3_levers.py": "feb247d2d08c52ba", "p2_scalars.py": "11feeaf3ba95ced4", "e22_test_oneshot.py": "669350bbfaaf0f4e", "e23_test_look7.py": "88587668baf12a64", "x1_external.py": "9962e3e7fe4ddc35", "p2_dump_seeds23.py": "0bc887542989310f", "e18_cpd_localizer.py": "e18c0ffe37f78e77", "e10_kernel_cpd.py": "5c692e33d92fd0fb", "dataprep/preprocessing.py": "0172fbea61b28540", "continuous_eval.py": "8fea38dabee5ad25", "c1b_build_trainset.py": "b4596a8aa3797653", "evaluation_protocol.py": "7caacac9b31bd0f7"}
sha = lambda p: hashlib.sha256(open(p, "rb").read()).hexdigest()[:16]
bad = {n: sha(f"{REPO}/src/{n}") for n in WANT if sha(f"{REPO}/src/{n}") != WANT[n]}
assert not bad, f"modules differ from the reviewed code (rebuild the notebook or pin the right commit): {bad}"
os.environ.update(THESIS_REPO=REPO, R2_WORK=WORK, R2_INPUT=INPUT, R2_SCRATCH=SCRATCH, OMP_NUM_THREADS="1",
                  OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="1")
if TEST:
    os.environ["R2_TEST"] = "1"
sys.path[:0] = [REPO + "/src"]
import eur_9a as M
assert M.__file__ == f"{REPO}/src/eur_9a.py", M.__file__
print("CPU cores:", os.cpu_count(), "| eur_9a", M.sha16(M.__file__), "| cases", len(M.CASES), "| smoke", M.SMOKE_CASE)

In [ ]:
# RUN: G-raw (PhysioNet), G-data, SMOKE (G-stats0, G-feat0), V5 control (R1.0, G-k), PROJ, label-free build
# (G-struct), scoring (k recalibrated; record k)
ERR = None
try:
    M.main(NOTEBOOK, HEAD, workers=int(os.environ.get("EUR_WORKERS", "0")) or None)
except SystemExit as e:
    ERR = str(e)

In [ ]:
# END: gates, projection, outputs, measured runtime
OUT = os.path.join(WORK, "eureka", "9a")
for f in ("gates.json", "projection.json", "run_meta.json", "r13/lf/k.json"):
    p = os.path.join(OUT, f)
    if os.path.exists(p):
        print("=====", f); print(open(p).read()[:4000])
for root, _, fs in os.walk(OUT):
    for f in sorted(fs):
        p = os.path.join(root, f); print(os.path.relpath(p, WORK), os.path.getsize(p))
print(f"MEASURED notebook runtime: {(time.time() - T_NB) / 3600:.3f} h")
if ERR:
    raise RuntimeError(f"STOPPED: {ERR}")